# Homework 2: Regressão linear
ML Zoomcamp 2026 | Car Fuel Efficiency

[Versão original no Google Colab](https://colab.research.google.com/drive/1g77auWHbpZQ9Y8i8j-U3V56J_JPF1DDy?usp=sharing)

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('car_fuel_efficiency_2026.csv')
cols = ['engine_displacement', 'horsepower', 'vehicle_weight',
        'model_year', 'fuel_efficiency_mpg']
df = df[cols]

## EDA

In [ ]:
sns.histplot(df.fuel_efficiency_mpg, bins=50)
plt.show()

## Q1 e Q2
**Q1:** `horsepower` (877 NAs) | **Q2:** 254

In [ ]:
print(df.isnull().sum())
print(df.horsepower.median())

## Funções auxiliares

In [ ]:
def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    return df_train, df_val, df_test

def train_linear_regression(X, y, r=0.0):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w[0], w[1:]

def predict(X, w0, w):
    return w0 + X.dot(w)

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

def prepare_X(df, fill_value):
    return df[features].fillna(fill_value).values

na_col = df.columns[df.isnull().any()][0]
print('Coluna com NAs:', na_col)

## Q3: preencher com 0 ou com a média
**Resposta:** com média (2.202 vs 2.205)

In [ ]:
df_train, df_val, df_test = split_data(df, seed=42)
y_train = df_train[target].values
y_val = df_val[target].values

X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)
w0, w = train_linear_regression(X_train, y_train)
score_0 = round(rmse(y_val, predict(X_val, w0, w)), 3)

mean_val = df_train[na_col].mean()  # média só do treino
X_train = prepare_X(df_train, {na_col: mean_val})
X_val = prepare_X(df_val, {na_col: mean_val})
w0, w = train_linear_regression(X_train, y_train)
score_mean = round(rmse(y_val, predict(X_val, w0, w)), 3)

print('Com 0:', score_0, '| Com média:', score_mean)

## Q4: regularização
**Resposta:** r = 0 (RMSE 2.2053)

In [ ]:
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression(X_train, y_train, r=r)
    print(r, round(rmse(y_val, predict(X_val, w0, w)), 4))

## Q5: influência da seed
**Resposta:** 0.029

In [ ]:
scores = []
for seed in range(10):
    tr, va, te = split_data(df, seed)
    Xtr, Xva = prepare_X(tr, 0), prepare_X(va, 0)
    w0, w = train_linear_regression(Xtr, tr[target].values)
    scores.append(rmse(va[target].values, predict(Xva, w0, w)))

print(round(np.std(scores), 3))

## Q6: RMSE no teste (seed 9, r = 0.001)
**Resposta:** 2.236

In [ ]:
tr, va, te = split_data(df, seed=9)
df_full = pd.concat([tr, va])

X_full = prepare_X(df_full, 0)
y_full = df_full[target].values
X_test = prepare_X(te, 0)
y_test = te[target].values

w0, w = train_linear_regression(X_full, y_full, r=0.001)
print(round(rmse(y_test, predict(X_test, w0, w)), 3))